# Diffusion-Pipe — Lium 8×H100 Training Run
Trainer: [`RicemanT/diffusion-pipe-mageflow-ft`](https://github.com/RicemanT/diffusion-pipe-mageflow-ft) @ `refresh`

### Run order on a fresh pod
`1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10` in order. Every cell is resumable and logs progress; nothing runs silently.

### Storage
| Path | What | Survives pod delete |
|---|---|---|
| `/workspace` | trainer code, Mage-Flow model | no |
| `/root` | dataset sidecars, latent cache | no (but larger + fast) |
| `/mnt` | configs, training outputs | **yes** (external Volume) |

### Why no images
The latent cache is pre-built. With `--trust_cache`, `cache_metadata()` loads `metadata/grouped_metadata_*` straight from disk and never enumerates the dataset directory, and `__getitem__` reads only cached latents. Tags captions are baked into `metadata.arrow` at cache-build time. Only `_nl.txt` is read live (`_load_nl_caption`), so the ~80 MB sidecar bundle is all you need — not the 76 GB of images.

### Pod setup
Template `Pytorch (Cuda + DinD) - daturaai/pytorch` (confirmed working), Volume attached, **Enable Jupyter** ticked, Auto-Termination set.

### 1 — Workspace

In [1]:
import os, subprocess

WORKSPACE_ROOT = '/mnt'          # external Volume: configs + outputs (persistent)
LOCAL_ROOT     = '/workspace'    # container overlay: trainer + model
DATA_ROOT      = '/root'         # pod local volume: sidecars + latent cache

for p in (WORKSPACE_ROOT, LOCAL_ROOT, DATA_ROOT):
    os.makedirs(p, exist_ok=True)
    print(f'📁 {p}')
subprocess.run('df -h /mnt /workspace /root', shell=True)


📁 /mnt
📁 /workspace
📁 /root
Filesystem      Size  Used Avail Use% Mounted on
s3fs            256T     0  256T   0% /mnt
overlay         995G   69M  995G   1% /
/lium-cipher    1.9T  1.9G  1.9T   1% /root


CompletedProcess(args='df -h /mnt /workspace /root', returncode=0)

### 2 — Tokens
Set `HF_TOKEN` (and `GH_TOKEN`) as Environment Variables on the Lium template, or fill them in below. Saved to the Volume so you only type them once.

In [2]:
import os, subprocess
subprocess.run('pip install -U huggingface_hub hf_transfer --break-system-packages -q', shell=True)

TOKENS_FILE = f'{WORKSPACE_ROOT}/.tokens.env'

HF_TOKEN = os.environ.get('HF_TOKEN', '')
GH_TOKEN = os.environ.get('GH_TOKEN', '')      # classic PAT, no scopes needed
WANDB_API_KEY = os.environ.get('WANDB_API_KEY', '')

# HF_TOKEN = 'hf_...'
# GH_TOKEN = 'ghp_...'
# WANDB_API_KEY = 'wandb_...'   # only needed for tracker = 'wandb'.
# Prefer $WANDB_API_KEY or .tokens.env over hardcoding: anything written
# here lands in the notebook file and in any screenshot of this cell.

if not HF_TOKEN and os.path.exists(TOKENS_FILE):
    for line in open(TOKENS_FILE):
        k, _, v = line.strip().partition('=')
        if v and not globals().get(k):
            globals()[k] = v

with open(TOKENS_FILE, 'w') as f:
    for k in ('HF_TOKEN', 'GH_TOKEN', 'WANDB_API_KEY'):
        f.write(f'{k}={globals()[k]}\n')
        os.environ[k] = globals()[k]

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✅ HuggingFace')
else:
    print('❌ HF_TOKEN missing — cache/model downloads will fail')
print('✅ GH_TOKEN' if GH_TOKEN else '⚠️  GH_TOKEN missing — ComfyUI submodule may resolve to the wrong commit')
print('✅ WANDB_API_KEY' if WANDB_API_KEY else 'ℹ️  WANDB_API_KEY not set — fine for tracker = trackio/none')


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ HuggingFace
✅ GH_TOKEN
✅ WANDB_API_KEY


### 3 — GPUs

In [3]:
import subprocess
r = subprocess.run('nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader',
                   shell=True, capture_output=True, text=True)
lines = [l for l in r.stdout.strip().split('\n') if l]
print(f'🖥️  {len(lines)} GPU(s)')
for l in lines:
    print('   ' + l)
if not lines:
    print('❌ none detected')


🖥️  8 GPU(s)
   0, NVIDIA H100 80GB HBM3, 81559 MiB
   1, NVIDIA H100 80GB HBM3, 81559 MiB
   2, NVIDIA H100 80GB HBM3, 81559 MiB
   3, NVIDIA H100 80GB HBM3, 81559 MiB
   4, NVIDIA H100 80GB HBM3, 81559 MiB
   5, NVIDIA H100 80GB HBM3, 81559 MiB
   6, NVIDIA H100 80GB HBM3, 81559 MiB
   7, NVIDIA H100 80GB HBM3, 81559 MiB


### 4 — Trainer + submodules
Fetched as **tarballs**, not git — anonymous `git clone` of github.com is refused from Lium pods (`could not read Username`). Submodule commits are resolved from the parent tree via the API, so you get exactly what `git submodule update` would give.

All 9 submodules are pulled: `utils/patches.py` imports `hyvideo` (HunyuanVideo) and `comfy` (ComfyUI) unconditionally at startup.

`MODE`: `clean` wipes and refetches, `resume` fills in only what's missing, `skip` leaves an existing install alone.

In [4]:
import os, io, json, time, shutil, tarfile, urllib.request, urllib.error, configparser
from concurrent.futures import ThreadPoolExecutor, as_completed

OWNER_REPO  = 'RicemanT/diffusion-pipe-mageflow-ft'
BRANCH      = 'refresh'
INSTALL_DIR = f'{LOCAL_ROOT}/diffusion-pipe-mageflow-ft'
MODE        = 'clean'      # 'clean' | 'resume' | 'skip'
WORKERS     = 6

def _get(url, timeout=600):
    req = urllib.request.Request(url, headers={'User-Agent': 'curl/8',
                                               'Accept': 'application/vnd.github+json'})
    if os.environ.get('GH_TOKEN'):
        req.add_header('Authorization', f'Bearer {os.environ["GH_TOKEN"]}')
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

def _untar(data, dest):
    os.makedirs(dest, exist_ok=True)
    with tarfile.open(fileobj=io.BytesIO(data), mode='r:gz') as tf:
        ms = tf.getmembers()
        root = ms[0].name.split('/')[0]
        for m in ms:
            if m.name == root:
                continue
            m.name = m.name[len(root)+1:]
            if m.name:
                tf.extract(m, dest)

os.chdir(LOCAL_ROOT)
if MODE == 'clean' and os.path.exists(INSTALL_DIR):
    print(f'🧹 removing {INSTALL_DIR}', flush=True)
    shutil.rmtree(INSTALL_DIR)

if MODE == 'skip' and os.path.exists(INSTALL_DIR):
    print('⏭️  exists — skipping', flush=True)
else:
    if not os.path.exists(os.path.join(INSTALL_DIR, 'train.py')):
        print(f'📥 main repo ({BRANCH})...', flush=True)
        for i in range(1, 4):
            try:
                _untar(_get(f'https://codeload.github.com/{OWNER_REPO}/tar.gz/refs/heads/{BRANCH}'), INSTALL_DIR)
                print('   ✅ main repo', flush=True); break
            except Exception as e:
                print(f'   ⚠️ {i}/3 {e}', flush=True); time.sleep(5)
        else:
            raise RuntimeError('main repo fetch failed')

    os.chdir(INSTALL_DIR)
    cp = configparser.ConfigParser()
    cp.read_string(open('.gitmodules').read().replace('\t', ''))
    subs = {cp[s]['path']: cp[s]['url'].rstrip('/').removesuffix('.git') for s in cp.sections()}

    print('🔎 resolving pinned submodule commits...', flush=True)
    pinned = {}
    try:
        tree = json.loads(_get(f'https://api.github.com/repos/{OWNER_REPO}/git/trees/{BRANCH}?recursive=1', timeout=90))
        if isinstance(tree, dict) and tree.get('message'):
            raise RuntimeError(tree['message'])
        for e in tree.get('tree', []):
            if e.get('type') == 'commit' and e['path'] in subs:
                pinned[e['path']] = e['sha']
    except Exception as e:
        print(f'   ⚠️ {e}', flush=True)
    print(f'   {len(pinned)}/{len(subs)} pinned', flush=True)
    if len(pinned) < len(subs):
        print('   ⚠️ unpinned ones fall back to HEAD, which can be NEWER than the trainer', flush=True)
        print('      expects (a newer ComfyUI breaks on comfy_kitchen). Set GH_TOKEN.', flush=True)

    todo = [p for p in subs
            if not (os.path.isdir(os.path.join(INSTALL_DIR, p)) and os.listdir(os.path.join(INSTALL_DIR, p)))]
    print(f'📦 {len(subs)} submodules, {len(todo)} to fetch', flush=True)

    def fetch(path):
        orp = subs[path].split('github.com/')[-1]
        dest = os.path.join(INSTALL_DIR, path)
        sha = pinned.get(path)
        urls = ([f'https://codeload.github.com/{orp}/tar.gz/{sha}'] if sha else []) + [
            f'https://codeload.github.com/{orp}/tar.gz/refs/heads/main',
            f'https://codeload.github.com/{orp}/tar.gz/refs/heads/master']
        last = None
        for u in urls:
            for a in range(2):
                try:
                    d = _get(u); _untar(d, dest)
                    return f'✅ {path} ({len(d)/1e6:.0f} MB, {"pinned" if sha and sha in u else "HEAD"})'
                except urllib.error.HTTPError as e:
                    last = e; break
                except Exception as e:
                    last = e; time.sleep(4)
        return f'❌ {path}: {last}'

    if todo:
        with ThreadPoolExecutor(max_workers=WORKERS) as pool:
            for f in as_completed({pool.submit(fetch, p): p for p in todo}):
                print('   ' + f.result(), flush=True)

os.chdir(INSTALL_DIR)
ok = True
for label, rel in [('comfy   (ComfyUI)', 'submodules/ComfyUI/comfy'),
                   ('hyvideo (HunyuanVideo)', 'submodules/HunyuanVideo/hyvideo')]:
    e = os.path.isdir(os.path.join(INSTALL_DIR, rel))
    print(f'   {"✅" if e else "❌"} {label}', flush=True); ok &= e

att = os.path.join(INSTALL_DIR, 'submodules/ComfyUI/comfy/ldm/modules/attention.py')
if os.path.exists(att):
    bad = 'comfy_kitchen' in open(att).read()
    print(f'   {"❌" if bad else "✅"} ComfyUI version check (comfy_kitchen import: {bad}, want False)', flush=True)
    if bad:
        print('      Wrong ComfyUI commit. Set GH_TOKEN and re-run with MODE="clean".', flush=True)
    ok &= not bad

print('✅ trainer ready' if ok else '❌ fix the above before continuing', flush=True)


📥 main repo (refresh)...
   ✅ main repo
🔎 resolving pinned submodule commits...
   9/9 pinned
📦 9 submodules, 9 to fetch
   ✅ submodules/flow (0 MB, pinned)
   ✅ submodules/LTX_Video (0 MB, pinned)
   ✅ submodules/HiDream (3 MB, pinned)
   ✅ submodules/Cosmos (9 MB, pinned)
   ✅ submodules/HunyuanVideo (48 MB, pinned)
   ✅ submodules/Lumina_2 (41 MB, pinned)
   ✅ submodules/OmniGen2 (76 MB, pinned)
   ✅ submodules/HunyuanImage-2.1 (6 MB, pinned)
   ✅ submodules/ComfyUI (7 MB, pinned)
   ✅ comfy   (ComfyUI)
   ✅ hyvideo (HunyuanVideo)
   ✅ ComfyUI version check (comfy_kitchen import: False, want False)
✅ trainer ready


### 5 — CUDA compiler
The template ships torch built for CUDA 13.0 but **no toolkit** — no `nvcc`, no `/usr/local/cuda*`. DeepSpeed's op-builder needs it or `import deepspeed` raises `MissingCUDAException`.

Installs only the compiler + headers (~hundreds of MB), not the ~5 GB toolkit. Version must match `torch.version.cuda`.

In [5]:
import os, glob, subprocess

def sh(c):
    print(f'  → {c}', flush=True)
    return subprocess.run(c, shell=True)

if glob.glob('/usr/local/cuda*/bin/nvcc'):
    print('✅ nvcc already installed', flush=True)
else:
    sh('apt-get update -qq')
    sh('apt-get install -y -qq wget')
    sh('wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/cuda-keyring_1.1-1_all.deb -O /tmp/ck.deb')
    sh('dpkg -i /tmp/ck.deb')
    sh('apt-get update -qq')
    sh('apt-get install -y -qq cuda-nvcc-13-0 cuda-cudart-dev-13-0')

cands = [c for c in sorted(glob.glob('/usr/local/cuda*')) if os.path.exists(f'{c}/bin/nvcc')]
if cands:
    os.environ['CUDA_HOME'] = cands[-1]
    os.environ['PATH'] = f'{cands[-1]}/bin:' + os.environ.get('PATH', '')
    print(f'✅ CUDA_HOME = {cands[-1]}', flush=True)
    sh(f'{cands[-1]}/bin/nvcc --version | tail -2')
else:
    print('❌ nvcc still not found', flush=True)


  → apt-get update -qq
  → apt-get install -y -qq wget


debconf: delaying package configuration, since apt-utils is not installed


(Reading database ... 24605 files and directories currently installed.)
Preparing to unpack .../wget_1.21.4-1ubuntu4.5_amd64.deb ...
Unpacking wget (1.21.4-1ubuntu4.5) over (1.21.4-1ubuntu4.4) ...
Setting up wget (1.21.4-1ubuntu4.5) ...
  → wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/cuda-keyring_1.1-1_all.deb -O /tmp/ck.deb
  → dpkg -i /tmp/ck.deb
Selecting previously unselected package cuda-keyring.
(Reading database ... 24605 files and directories currently installed.)
Preparing to unpack /tmp/ck.deb ...
Unpacking cuda-keyring (1.1-1) ...
Setting up cuda-keyring (1.1-1) ...
  → apt-get update -qq
  → apt-get install -y -qq cuda-nvcc-13-0 cuda-cudart-dev-13-0


debconf: delaying package configuration, since apt-utils is not installed


Selecting previously unselected package cuda-cccl-13-0.
(Reading database ... 24610 files and directories currently installed.)
Preparing to unpack .../00-cuda-cccl-13-0_13.0.85-1_amd64.deb ...
Unpacking cuda-cccl-13-0 (13.0.85-1) ...
Selecting previously unselected package cuda-toolkit-config-common.
Preparing to unpack .../01-cuda-toolkit-config-common_13.3.29-1_all.deb ...
Unpacking cuda-toolkit-config-common (13.3.29-1) ...
Selecting previously unselected package cuda-toolkit-13-config-common.
Preparing to unpack .../02-cuda-toolkit-13-config-common_13.3.29-1_all.deb ...
Unpacking cuda-toolkit-13-config-common (13.3.29-1) ...
Selecting previously unselected package cuda-toolkit-13-0-config-common.
Preparing to unpack .../03-cuda-toolkit-13-0-config-common_13.0.96-1_all.deb ...
Unpacking cuda-toolkit-13-0-config-common (13.0.96-1) ...
Selecting previously unselected package cuda-cudart-13-0.
Preparing to unpack .../04-cuda-cudart-13-0_13.0.96-1_amd64.deb ...
Unpacking cuda-cudart-13

### 6 — Directories

In [6]:
import os
DATASET_DIR = f'{DATA_ROOT}/datasets/Booru-Essence-2026'   # must match dataset.toml `path`
for d in [f'{LOCAL_ROOT}/Mage-Flow', DATASET_DIR,
          f'{WORKSPACE_ROOT}/outputs', f'{WORKSPACE_ROOT}/configs']:
    os.makedirs(d, exist_ok=True)
    print(f'📁 {d}')
print('\n⚠️  DATASET_DIR must stay byte-identical to when the cache was built —')
print('   the fingerprint hashes these absolute paths.')


📁 /workspace/Mage-Flow
📁 /root/datasets/Booru-Essence-2026
📁 /mnt/outputs
📁 /mnt/configs

⚠️  DATASET_DIR must stay byte-identical to when the cache was built —
   the fingerprint hashes these absolute paths.


### 7 — Mage-Flow model
Self-contained diffusers repo (own `transformer/`, `vae/`, `text_encoder/`, `model_index.json`) onto pod-local disk. Config `diffusers_path` should point here.

In [7]:
import os, time
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download

DEST = f'{LOCAL_ROOT}/Mage-Flow'
have = [f for r, _, fs in os.walk(DEST) for f in fs if not f.startswith('.')]
if have:
    print(f'⏭️  already present ({len(have)} files)', flush=True)
else:
    t = time.time()
    snapshot_download(repo_id='mage-flow-community/Mage-Flow', local_dir=DEST,
                      token=os.environ['HF_TOKEN'], max_workers=8,
                      ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'])
    print(f'✅ {(time.time()-t)/60:.1f} min', flush=True)
print(sorted(os.listdir(DEST))[:12])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 43 files:   0%|          | 0/43 [00:00<?, ?it/s]

✅ 2.3 min
['.cache', '.gitattributes', 'README.md', 'assets', 'model_index.json', 'scheduler', 'text_encoder', 'transformer', 'vae']


### 8 — Caption sidecars
Only `_nl.txt` is read at runtime (`_load_nl_caption`); tags are already baked into the cached `metadata.arrow`. Both are shipped anyway — ~80 MB total, so the 76 GB of images are never needed on this pod.

Falls back to rebuilding from the source tars if the bundle isn't on the Hub.

In [8]:
import os, time, subprocess, shutil
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import hf_hub_download

CACHE_REPO   = 'RicemanT/booru-essence-mage-latents'
SIDECAR_FILE = 'sidecars.tar.gz'

n_txt = sum(1 for f in os.listdir(DATASET_DIR) if f.endswith('.txt')) if os.path.isdir(DATASET_DIR) else 0
if n_txt > 1000:
    print(f'⏭️  {n_txt} sidecars already present', flush=True)
else:
    t = time.time()
    print(f'📥 {CACHE_REPO}/{SIDECAR_FILE}', flush=True)
    p = hf_hub_download(repo_id=CACHE_REPO, filename=SIDECAR_FILE, repo_type='model',
                        token=os.environ['HF_TOKEN'], local_dir='/tmp/sidecars')

    if not shutil.which('pigz'):
        subprocess.run('apt-get install -y -qq pigz', shell=True)

    # GNU tar in C, not Python's tarfile: 83k tiny files is pure per-file
    # overhead, and the pure-Python loop runs ~290 files/s. --no-same-owner
    # skips a chown syscall per file; pigz parallelizes decompression.
    decomp = '--use-compress-program=pigz' if shutil.which('pigz') else '-z'
    print(f'📦 extracting {os.path.getsize(p)/1e6:.0f} MB with tar {decomp}...', flush=True)
    r = subprocess.run(f'tar {decomp} -xf "{p}" -C "{DATASET_DIR}" --no-same-owner',
                       shell=True)
    if r.returncode != 0:
        print('   ⚠️ tar failed, falling back to Python tarfile', flush=True)
        import tarfile
        with tarfile.open(p, 'r:gz') as tf:
            tf.extractall(DATASET_DIR)

    os.remove(p)
    n_txt = sum(1 for f in os.listdir(DATASET_DIR) if f.endswith('.txt'))
    print(f'✅ {n_txt} sidecars in {(time.time()-t)/60:.1f} min', flush=True)

nl = sum(1 for f in os.listdir(DATASET_DIR) if f.endswith('_nl.txt'))
print(f'   {nl} _nl.txt  |  {n_txt-nl} tag .txt', flush=True)

📥 RicemanT/booru-essence-mage-latents/sidecars.tar.gz


sidecars.tar.gz: reconstructing file:   0%|          |  0.00B / 45.3MB            

sidecars.tar.gz: downloading bytes:           |  0.00B            

debconf: delaying package configuration, since apt-utils is not installed


Selecting previously unselected package pigz.
(Reading database ... 26639 files and directories currently installed.)
Preparing to unpack .../archives/pigz_2.8-1_amd64.deb ...
Unpacking pigz (2.8-1) ...
Setting up pigz (2.8-1) ...
📦 extracting 45 MB with tar --use-compress-program=pigz...
✅ 83196 sidecars in 5.1 min
   41598 _nl.txt  |  41598 tag .txt


### 9 — Latent cache
Downloads the pre-built VAE latents + `metadata/` into `{DATASET_DIR}/cache/mage_flow/`. `metadata/` is what `--trust_cache` reads — without it the trainer re-enumerates the dataset dir and fails with no images present.

Resumable: re-running skips what's already local.

In [9]:
import os, time
# Must be set BEFORE huggingface_hub is imported. This repo is Xet-backed
# (the 'xet' badge on the HF page), so these govern the transfer, not hf_transfer.
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'
os.environ['HF_XET_NUM_CONCURRENT_RANGE_GETS'] = '32'

from huggingface_hub import snapshot_download

CACHE_REPO = 'RicemanT/booru-essence-mage-latents'
DEST = os.path.join(DATASET_DIR, 'cache', 'mage_flow')
os.makedirs(DEST, exist_ok=True)

def gb(p):
    return sum(os.path.getsize(os.path.join(r, f))
               for r, _, fs in os.walk(p) for f in fs) / 1e9

if gb(DEST) > 40:
    print(f'⏭️  cache already present ({gb(DEST):.1f} GB)', flush=True)
else:
    t = time.time()
    print(f'📥 {CACHE_REPO} → {DEST}', flush=True)
    snapshot_download(repo_id=CACHE_REPO, repo_type='model', local_dir=DEST,
                      token=os.environ['HF_TOKEN'], max_workers=32,
                      ignore_patterns=['sidecars.tar.gz'])
    el = (time.time() - t) / 60
    print(f'✅ {gb(DEST):.1f} GB in {el:.1f} min ({gb(DEST)*1000/(el*60):.0f} MB/s)', flush=True)

subs = sorted(d for d in os.listdir(DEST) if os.path.isdir(os.path.join(DEST, d)))
print(f'   {len(subs)} dirs, metadata {"✅" if "metadata" in subs else "❌ MISSING"}', flush=True)

📥 RicemanT/booru-essence-mage-latents → /root/datasets/Booru-Essence-2026/cache/mage_flow


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 116 files:   0%|          | 0/116 [00:00<?, ?it/s]

✅ 44.1 GB in 5.3 min (138 MB/s)
   16 dirs, metadata ✅


### 10 — Training
Installs deps fresh (the container resets every pod), then launches DeepSpeed with `--trust_cache` so metadata loads from the cache and the dataset dir is never enumerated.

Set `CACHE_ONLY = True` to build a cache and exit instead of training.

In [13]:
import os, re, sys, glob, time, subprocess, importlib, tempfile
from tqdm.auto import tqdm

REPO_DIR    = f'{LOCAL_ROOT}/diffusion-pipe-mageflow-ft'
CONFIG_PATH = f'{WORKSPACE_ROOT}/configs/mage_flow_BooruEssenceFFT.toml'
TRAIN_SCRIPT = f'{REPO_DIR}/train.py'

CACHE_ONLY       = False    # True = build latent cache then exit
TRUST_CACHE      = True     # skip metadata re-enumeration (required when images absent)
DISABLE_NCCL_P2P = False
CAPTION_DEBUG    = 'collapse'   # 'collapse' | 'full' | 'hide'
MAX_LINE_CHARS   = 400

for p, what in [(REPO_DIR, 'trainer (cell 4)'), (CONFIG_PATH, 'config'), (TRAIN_SCRIPT, 'train.py')]:
    if not os.path.exists(p):
        print(f'❌ missing {what}: {p}'); raise SystemExit

os.chdir(REPO_DIR)
PIP = '--break-system-packages'
GIT = 'GIT_TERMINAL_PROMPT=0'

# Which experiment tracker the config asks for: 'wandb' | 'trackio' | 'none'.
# Falls back to the legacy enable_wandb key when [monitoring] tracker is absent.
_cfg_text = open(CONFIG_PATH).read()
_m = re.search(r'^\s*tracker\s*=\s*[\'"](\w+)[\'"]', _cfg_text, re.M)
if _m:
    TRACKER = _m.group(1).lower()
else:
    _m = re.search(r'^\s*enable_wandb\s*=\s*(\w+)', _cfg_text, re.M)
    TRACKER = 'wandb' if (_m and _m.group(1).lower() == 'true') else 'none'
print(f'📊 tracker: {TRACKER}', flush=True)

def run(c, check=False):
    return subprocess.run(c, shell=True, check=check)
def pin(pkg, extra=''):
    r = run(f'{GIT} {sys.executable} -m pip install "{pkg}" {extra} {PIP} -q')
    if r.returncode: print(f'   ⚠️  {pkg} failed', flush=True)
def pun(pkg):
    run(f'{sys.executable} -m pip uninstall {pkg} -y {PIP} -q')   # needs the flag too

# --- CUDA first: importing deepspeed builds ops and reads CUDA_HOME ---
print('🔧 CUDA...', flush=True)
if 'CUDA_HOME' not in os.environ:
    c = [x for x in sorted(glob.glob('/usr/local/cuda*')) if os.path.exists(f'{x}/bin/nvcc')]
    if c: os.environ['CUDA_HOME'] = c[-1]
cuda_home = os.environ.get('CUDA_HOME', '')
if not cuda_home or not os.path.exists(f'{cuda_home}/bin/nvcc'):
    print('❌ nvcc missing — run cell 5'); raise SystemExit
os.environ['PATH'] = f'{cuda_home}/bin:' + os.environ.get('PATH', '')
print(f'   {cuda_home}', flush=True)
# torch caches CUDA_HOME at import; patch it in case torch was imported earlier
try:
    import torch.utils.cpp_extension as _ce
    if _ce.CUDA_HOME != cuda_home:
        _ce.CUDA_HOME = cuda_home
        print('   (patched torch cpp_extension)', flush=True)
except Exception:
    pass

print('⚙️  system deps...', flush=True)
run('apt-get update -qq && apt-get install -y libaio-dev libcufile-dev -qq')

try:
    import torch
    print(f'🔥 torch {torch.__version__} (cuda {torch.version.cuda}) — keeping', flush=True)
except ImportError:
    run(f'{sys.executable} -m pip install torch torchvision {PIP} -q')

# stage-lr is a git dependency; anonymous git is refused from these pods, so
# install it from the codeload tarball instead (default branch is master).
print('📋 requirements.txt (minus stage-lr)...', flush=True)
lines = [l for l in open(f'{REPO_DIR}/requirements.txt') if 'stage-lr' not in l and 'stage_lr' not in l]
with tempfile.NamedTemporaryFile('w', suffix='.txt', delete=False) as tf:
    tf.writelines(lines); req = tf.name
run(f'{GIT} {sys.executable} -m pip install -r {req} {PIP} -q')
os.unlink(req)

print('🔧 stage-lr...', flush=True)
try:
    importlib.import_module('stage_lr'); print('   ✅ present', flush=True)
except ImportError:
    for ref in ('master', 'main'):
        if run(f'{sys.executable} -m pip install "https://codeload.github.com/nruaif/stage-lr/tar.gz/refs/heads/{ref}" {PIP} -q').returncode == 0:
            print(f'   ✅ from {ref}', flush=True); break

print('🔧 extras...', flush=True)
for p in ['multiprocess', 'protobuf>=6.32.1,<7.0', 'tqdm', 'opencv-python-headless',
          'psutil', 'einops', 'kornia', 'spandrel', 'torchsde', 'soundfile',
          'aiohttp', 'pyyaml', 'scipy', 'comfy_aimdo']:
    pin(p)

if TRACKER == 'trackio':
    # Pinned: Trackio is pre-release and its SQLite schema can change.
    print('🔧 trackio...', flush=True)
    pin('trackio==0.37.0')

print('🔧 torchao + deepspeed (pinned)...', flush=True)
pun('torchao');   pin('torchao==0.16.0')
pun('deepspeed'); pin('deepspeed==0.18.4')
# fsspec last: `datasets` pulls a version it then rejects
pin('fsspec[http]<=2026.6.0', '--upgrade')

print('\n🔍 pip check:', flush=True); run(f'{sys.executable} -m pip check')

print('\n🔍 imports...', flush=True)
ok = True
_mods = ['torch', 'deepspeed', 'multiprocess', 'datasets', 'diffusers',
         'transformers', 'accelerate', 'bitsandbytes', 'cv2']
if TRACKER in ('wandb', 'trackio'):
    _mods.append(TRACKER)
for m in _mods:
    try:
        importlib.import_module(m); print(f'   ✅ {m}', flush=True)
    except Exception as e:
        print(f'   ❌ {m} — {type(e).__name__}: {e}', flush=True); ok = False
try:
    from transformers import Qwen3VLForConditionalGeneration
    import transformers
    print(f'   ✅ Qwen3VL (transformers {transformers.__version__})', flush=True)
except Exception as e:
    print(f'   ❌ Qwen3VL — {e}', flush=True); ok = False
try:
    sys.path.insert(0, f'{REPO_DIR}/submodules/ComfyUI')
    importlib.import_module('comfy.model_management'); print('   ✅ comfy', flush=True)
except Exception as e:
    print(f'   ❌ comfy — {e}', flush=True); ok = False
if re.search(r'^\s*lr_scheduler\s*=\s*["\']StageLR["\']', open(CONFIG_PATH).read(), re.M):
    try:
        importlib.import_module('stage_lr'); print('   ✅ stage_lr', flush=True)
    except Exception as e:
        print(f'   ❌ stage_lr (config needs it) — {e}', flush=True); ok = False
if not ok:
    print('\n❌ fix the above'); raise SystemExit
print('✅ deps ready\n', flush=True)

g = subprocess.run('nvidia-smi --query-gpu=index --format=csv,noheader',
                   shell=True, capture_output=True, text=True).stdout.strip().splitlines()
if not g:
    print('❌ no GPUs'); raise SystemExit
idx = ','.join(x.strip() for x in g)
print(f'🖥️  {len(g)} GPU(s): {idx}', flush=True)

env = f'CUDA_HOME={cuda_home} PATH={cuda_home}/bin:$PATH PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True '
if DISABLE_NCCL_P2P:
    env += 'NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 '

flags = ' --cache_only' if CACHE_ONLY else ''
flags += ' --trust_cache' if TRUST_CACHE else ''
cmd = (f'cd {REPO_DIR} && {env}{sys.executable} -m deepspeed.launcher.runner '
       f'--include localhost:{idx} {TRAIN_SCRIPT} --deepspeed --config {CONFIG_PATH}{flags}')
print(f'\n🚀 {cmd}\n' + '='*60, flush=True)

ds_step = re.compile(r"\[Rank \d+\]\s+step=(?P<step>\d+),\s+skipped=\d+,\s+lr=\[(?P<lr>[^\]]+)\]")
thru    = re.compile(r"steps:\s+(?P<step>\d+)\s+loss:\s+(?P<loss>[\d.]+|nan|inf|-inf)\s+iter time.*?samples/sec:\s+(?P<speed>[\d.]+)")
epoch_p = re.compile(r"Started new epoch:\s*(?P<epoch>\d+)")
steplog = re.compile(r"^epoch:\s+(?P<epoch>\d+)\s+step:\s+(?P<step>\d+)\s+lr:\s+(?P<lr>n/a|[\d.]+e[+-]\d+(?:\s*/\s*[\d.]+e[+-]\d+)*)\s+loss:\s+(?P<loss>[\d.]+|nan|inf|-inf)(?:\s+grad_norm:\s+(?P<gn>[\d.]+))?$")
sample_p= re.compile(r"Generated\s+(?P<ok>\d+)/(?P<total>\d+)\s+sample\(s\)\s+in\s+(?P<secs>[\d.]+)s")
cap_s   = re.compile(r"^\[Caption Debug \| Sample (?P<n>\d+)\]")
cap_b   = re.compile(r"^[├└]─")
spam = ["FutureWarning: _check_is_size", "torch._check_is_size(blocksize)", "_guard_size_oblivious",
        "bitsandbytes/_ops.py", "Skipping import of cpp extensions due to incompatible torch version"]

def flr(x):
    try: return f"{float(x.split(',')[0]):.3e}"
    except ValueError: return x.strip()

proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
pbar = tqdm(total=None, desc='Epoch ?', unit='step', dynamic_ncols=True, mininterval=0.3)
post, in_cap, cap_n = {}, False, None
try:
    for line in iter(proc.stdout.readline, ''):
        L = line.strip()
        if not L or any(s in L for s in spam):
            continue
        m = cap_s.search(L)
        if m:
            in_cap, cap_n = True, m['n']
            if CAPTION_DEBUG == 'full': tqdm.write(L)
            continue
        if in_cap:
            if cap_b.search(L):
                if CAPTION_DEBUG == 'full': tqdm.write(L)
                elif CAPTION_DEBUG == 'collapse' and L.startswith('└─'):
                    tqdm.write(f'[Caption Debug] sample {cap_n} (collapsed)')
                continue
            in_cap = False
        m = ds_step.search(L)
        if m:
            pbar.n = pbar.last_print_n = int(m['step'])
            post['LR'] = flr(m['lr']); pbar.set_postfix(post); pbar.refresh(); continue
        m = thru.search(L)
        if m:
            pbar.n = pbar.last_print_n = max(pbar.n, int(m['step']))
            post['Loss'] = m['loss']; post['Img/s'] = f"{float(m['speed']):.2f}"
            pbar.set_postfix(post); pbar.refresh(); continue
        m = steplog.search(L)
        if m:
            post['LR'], post['Loss'] = m['lr'], m['loss']
            if m['gn']: post['GradNorm'] = m['gn']
            pbar.set_postfix(post); pbar.refresh(); continue
        m = epoch_p.search(L)
        if m:
            pbar.set_description(f"Epoch {m['epoch']}"); tqdm.write(L); continue
        m = sample_p.search(L)
        if m:
            post['Samples'] = f"{m['ok']}/{m['total']}"; pbar.set_postfix(post); tqdm.write(L); continue
        if 'Training Progress:' in L:
            continue
        tqdm.write(L if len(L) <= MAX_LINE_CHARS else L[:MAX_LINE_CHARS] + ' …')
except KeyboardInterrupt:
    tqdm.write('\n🔴 interrupted'); proc.terminate()
finally:
    pbar.close()
    if proc.poll() == 0:
        tqdm.write('\n✅ finished')


📊 tracker: none
🔧 CUDA...
   /usr/local/cuda-13.0
⚙️  system deps...
🔥 torch 2.12.0+cu130 (cuda 13.0) — keeping
📋 requirements.txt (minus stage-lr)...
🔧 stage-lr...
   ✅ present
🔧 extras...


🔧 torchao + deepspeed (pinned)...



🔍 pip check:


No broken requirements found.

🔍 imports...
   ✅ torch
   ✅ deepspeed
   ✅ multiprocess
   ✅ datasets
   ✅ diffusers
   ✅ transformers
   ✅ accelerate
   ✅ bitsandbytes
   ✅ cv2
   ✅ Qwen3VL (transformers 5.16.1)
   ✅ comfy
   ✅ stage_lr
✅ deps ready

🖥️  8 GPU(s): 0,1,2,3,4,5,6,7

🚀 cd /workspace/diffusion-pipe-mageflow-ft && CUDA_HOME=/usr/local/cuda-13.0 PATH=/usr/local/cuda-13.0/bin:$PATH PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True /usr/bin/python -m deepspeed.launcher.runner --include localhost:0,1,2,3,4,5,6,7 /workspace/diffusion-pipe-mageflow-ft/train.py --deepspeed --config /mnt/configs/mage_flow_BooruEssenceFFT.toml --trust_cache


Epoch ?: 0step [00:00, ?step/s]

[2026-09-04 10:55:12,796] [WARNING] [runner.py:232:fetch_hostfile] Unable to find hostfile, will proceed with training with local resources only.
[2026-09-04 10:55:12,796] [INFO] [runner.py:630:main] cmd = /usr/bin/python -u -m deepspeed.launcher.launch --world_info=eyJsb2NhbGhvc3QiOiBbMCwgMSwgMiwgMywgNCwgNSwgNiwgN119 --master_addr=127.0.0.1 --master_port=29500 --enable_each_rank_log=None --log_level=info /workspace/diffusion-pipe-mageflow-ft/train.py --deepspeed --config /mnt/configs/mage_flow_BooruEssenceFFT.toml --trust_cache
[2026-09-04 10:55:17,443] [INFO] [launch.py:162:main] WORLD INFO DICT: {'localhost': [0, 1, 2, 3, 4, 5, 6, 7]}
[2026-09-04 10:55:17,443] [INFO] [launch.py:168:main] nnodes=1, num_local_procs=8, node_rank=0
[2026-09-04 10:55:17,443] [INFO] [launch.py:179:main] global_rank_mapping=defaultdict(<class 'list'>, {'localhost': [0, 1, 2, 3, 4, 5, 6, 7]})
[2026-09-04 10:55:17,443] [INFO] [launch.py:180:main] dist_world_size=8
[2026-09-04 10:55:17,443] [INFO] [launch.py:

### 10b — Trackio dashboard (optional)

Only needed when the config has `tracker = 'trackio'` **and** you are using
local mode (no `trackio_space_id` / `trackio_server_url`).

Run this in a **separate** notebook/kernel while training is going — it blocks.
Then from your own machine:

```
ssh -L 7860:localhost:7860 <user>@<pod-host>
```

and open <http://localhost:7860>. Live metrics and validation sample images,
with no dependency on any hosted service being reachable from the pod.


In [ ]:
# Standalone: run this in its OWN notebook/kernel while training runs in the
# other one. It does not depend on any variable from the training cell.
import os, re, glob, sys

# --- edit if yours differ ---------------------------------------------
WORKSPACE_ROOT = '/mnt'
CONFIG_PATH    = f'{WORKSPACE_ROOT}/configs/mage_flow_BooruEssenceFFT.toml'
# ----------------------------------------------------------------------

if not os.path.exists(CONFIG_PATH):
    print(f'❌ config not found: {CONFIG_PATH}')
    raise SystemExit

cfg = open(CONFIG_PATH).read()
def _get(pat, default=None):
    m = re.search(pat, cfg, re.M)
    return m.group(1) if m else default

OUTPUT_DIR = _get(r'^\s*output_dir\s*=\s*[\'"]([^\'"]+)')
PROJECT    = _get(r'^\s*wandb_tracker_name\s*=\s*[\'"]([^\'"]+)', 'diffusion-pipe')
TRACKER    = _get(r'^\s*tracker\s*=\s*[\'"](\w+)[\'"]', 'none').lower()

if TRACKER != 'trackio':
    print(f"ℹ️  config has tracker = '{TRACKER}', not 'trackio' — nothing to show here.")
elif not OUTPUT_DIR:
    print('❌ could not read output_dir from the config')
else:
    # The trainer writes to <run_dir>/trackio (see utils/tracking.py).
    runs = sorted(glob.glob(f'{OUTPUT_DIR}/*/trackio'))
    if not runs:
        print(f'❌ no trackio data under {OUTPUT_DIR}/*/trackio')
        print('   Start training first. If the trainer printed a path under')
        print('   ~/.cache/huggingface/trackio instead, your trainer copy predates')
        print('   the TRACKIO_DIR fix — git pull.')
    else:
        os.environ['TRACKIO_DIR'] = runs[-1]   # must be set before trackio imports
        print(f'📊 {runs[-1]}')
        print(f'   project: {PROJECT}')
        print('   tunnel:  ssh -L 7860:localhost:7860 <user>@<pod-host> -p <port>')
        print('   then open http://localhost:7860   (Ctrl+C here to stop)\n', flush=True)
        # trackio has no `python -m trackio` entry point (no __main__.py),
        # so the dashboard is started via trackio.show(), not a subprocess CLI
        # call. It binds 127.0.0.1:7860 by default and blocks this cell until
        # interrupted -- that block IS the running state, not a hang.
        import trackio
        trackio.show(project=PROJECT, host='127.0.0.1', open_browser=False)

### 11 — Upload outputs
Checkpoints are written to `output_dir` on the Volume, so they already survive pod deletion. This is for sharing / offsite backup.

In [15]:
import os, glob, time
from huggingface_hub import HfApi, create_repo

# ---- edit ----------------------------------------------------------------
HF_UPLOAD_REPO = 'RicemanT/MageFlow-Stuffs'   # MUST include your username
PRIVATE        = False
RUN_GLOB       = '/workspace/outputs/MageFlow-Booru-Essence-2026-continue/*'
UPLOAD_GLOBAL_STEP = True   # False = skip the DeepSpeed resume state (much faster)
# --------------------------------------------------------------------------

runs = sorted(glob.glob(RUN_GLOB))
if not runs:
    raise SystemExit(f'❌ no run dir matched {RUN_GLOB}')
RUN = runs[-1]
print(f'📂 run: {RUN}')

targets = []          # (local_folder, path_in_repo)

ep = os.path.join(RUN, 'epoch10')
if os.path.isdir(ep):
    # config.json is required to load this later via transformer_path --
    # the trainer saves the .toml but never a config.json.
    src_cfg = f'{WORKSPACE_ROOT}/Mage-Flow/transformer/config.json'
    if os.path.exists(src_cfg) and not os.path.exists(os.path.join(ep, 'config.json')):
        import shutil; shutil.copy(src_cfg, ep)
        print('   + copied config.json into epoch10')
    targets.append((ep, 'epoch30'))          # cumulative name: 20 + 10
else:
    print(f'⚠️  no epoch10 in {RUN}')

if UPLOAD_GLOBAL_STEP:
    for gs in sorted(glob.glob(os.path.join(RUN, 'global_step*'))):
        targets.append((gs, os.path.basename(gs)))

if not targets:
    raise SystemExit('❌ nothing to upload')

total = 0
for folder, dest in targets:
    size = sum(os.path.getsize(os.path.join(r, f))
               for r, _, fs in os.walk(folder) for f in fs)
    total += size
    print(f'   {size/1e9:6.2f} GB  {folder}  ->  {dest}/')
print(f'   {total/1e9:6.2f} GB total  (~{total/1e9/0.062/60:.0f} min at ~62 MB/s)')

api = HfApi(token=os.environ['HF_TOKEN'])
create_repo(HF_UPLOAD_REPO, token=os.environ['HF_TOKEN'],
            private=PRIVATE, exist_ok=True)

for i, (folder, dest) in enumerate(targets, 1):
    print(f'\n[{i}/{len(targets)}] ↑ {dest}/', flush=True)
    t = time.time()
    api.upload_folder(folder_path=folder, path_in_repo=dest,
                      repo_id=HF_UPLOAD_REPO, repo_type='model')
    print(f'    ✅ {(time.time()-t)/60:.1f} min', flush=True)

print(f'\n✅ https://huggingface.co/{HF_UPLOAD_REPO}')

📂 run: /workspace/outputs/MageFlow-Booru-Essence-2026-continue/20260904_10-56-34
     8.23 GB  /workspace/outputs/MageFlow-Booru-Essence-2026-continue/20260904_10-56-34/epoch10  ->  epoch30/
    25.67 GB  /workspace/outputs/MageFlow-Booru-Essence-2026-continue/20260904_10-56-34/global_step213  ->  global_step213/
    33.90 GB total  (~9 min at ~62 MB/s)

[1/2] ↑ epoch30/
    ✅ 1.4 min

[2/2] ↑ global_step213/
    ✅ 2.7 min

✅ https://huggingface.co/RicemanT/MageFlow-Stuffs
